In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import lightgbm as lgb


In [21]:


train = pd.read_csv('../../data/raw/train.csv',  )


/var/folders/5p/tsf09yfn1d7ct362cxy57hyc0000gn/T/ipykernel_12536/135321607.py:1: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../../data/raw/train.csv',  )


### I saved a sample of the data to a parquet file so we can all run this as a sanity check, before we scale up


In [22]:
slice = train.loc[train['date'] > '2017-02-01']
slice.to_parquet('../../data/sample/train_six_month_slice.parquet')

In [7]:
slice.shape
slice = slice.astype({
    "store_nbr": "int8",
    "item_nbr": "int32",
    "unit_sales": "float32",
    "onpromotion": "int8",
})

(7913096, 6)

In [ ]:
# v2: merge store metadata on every row; holidays table only for holiday flag
stores = pd.read_csv("../../data/raw/stores.csv")
stores = stores.rename(columns={"type": "store_type", "cluster": "store_cluster"})

holidays = pd.read_csv("../../data/cleaned/stores_holidays.csv")
holidays["date"] = pd.to_datetime(holidays["date"])
holidays_subset = (
    holidays.groupby(["date", "store_nbr"], as_index=False)["holiday"]
    .max()
)


def add_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    df = df.merge(
        stores[["store_nbr", "store_type", "store_cluster"]],
        on="store_nbr",
        how="left",
    )
    df = df.merge(holidays_subset, on=["date", "store_nbr"], how="left")

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)
    df["holiday"] = df["holiday"].fillna(False).astype(int)

    df["store_type"] = df["store_type"].astype("category")
    df["store_nbr"] = df["store_nbr"].astype("category")
    df["item_nbr"] = df["item_nbr"].astype("category")

    return df


feature_cols = [
    "store_nbr",
    "item_nbr",
    "store_type",
    "store_cluster",
    "year",
    "month",
    "day",
    "dayofweek",
    "onpromotion",
    "holiday",
]

cat_cols = ["store_nbr", "item_nbr", "store_type"]

# in training cell below, use:
# train = add_features(train)
# X_train = train[feature_cols]
# model.fit(X_train, y_train, categorical_feature=cat_cols)


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

train = pd.read_parquet("../../data/sample/train_two_month_slice.parquet")  # or your slice

y_train = train["unit_sales"]

# for training with added features
train = add_features(train)
X_train = train[feature_cols]


model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,      # primary complexity knob (not max_depth)
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
# model.fit(X_train, y_train)

# v2
model.fit(X_train, y_train, categorical_feature=cat_cols)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.139076 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3771
[LightGBM] [Info] Number of data points in the train set: 20564438, number of used features: 10
[LightGBM] [Info] Start training from score 8.117268


,num_leaves,63
,learning_rate,0.05
,n_estimators,1000
,min_child_samples,50
,subsample,0.8
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000


In [ ]:
test = pd.read_csv("../../data/raw/test.csv")
# test = add_features(test)
# X_test = test[feature_cols]

test = add_features(test)
X_test = test[feature_cols]

preds = model.predict(X_test)
# sales can't be negative
preds = preds.clip(min=0)
submission = pd.DataFrame({
    "id": test["id"],
    "unit_sales": preds,
})

In [25]:
sample = pd.read_csv("../../data/raw/sample_submission.csv")
assert list(submission.columns) == ["id", "unit_sales"]
assert len(submission) == len(sample) == 3_370_464
assert submission["id"].equals(sample["id"])  # same ids, same order
# print(submission.head())
# print(submission.describe())

In [ ]:
submission.to_csv("../../submissions/lgb_two_month_items_submission.csv", index=False)